# Cellular Automata Simulation of Granular Flow
## Capstone Project Notebook

A *cellular automaton* (CA) is a grid of cells, each of which holds one of a finite set of states. Time advances in discrete steps, and at each step every cell updates its state simultaneously according to a rule that depends only on its own state and the states of its nearby neighbours. Although each rule is simple and local, when repeated over many steps these updates can produce complex, large-scale patterns.

### Key Characteristics of Cellular Automata

- **Discrete space and time:** The grid has a fixed shape (e.g., square, hexagon) and cells update in lockstep at each time step.
- **Finite states:** Each cell can take one of a limited number of values (often just two, such as 0 or 1).
- **Local rules:** A cell's next state is determined by a fixed rule applied to its neighbourhood.
- **Parallel update:** All cells compute their next state at the same time, avoiding bias from sequential updates.

Early work by John von Neumann explored how CAs could model self-replication, and Stephen Wolfram classified one-dimensional "elementary" automata (256 possible rules) into four behaviour classes, from uniform to chaotic to complex. In two dimensions, Conway's Game of Life uses binary states and an eight-cell neighbourhood to generate stable patterns ("still lifes"), oscillators, and moving structures ("gliders").

### Granular Automaton vs. Real-World Granular Flow

In physics, granular flow is often modelled via discrete element methods (DEM) or continuum approaches, which resolve interparticle forces and contact dynamics. Our cellular automaton is an abstraction: each grid cell holds at most one particle, and simple neighbour-based rules drive movement and interaction. While less detailed than DEM, this approach highlights **emergent behaviour from minimal rules** and is computationally efficient for large grids.

In the indie game *Noita*, a similar CA framework is used to simulate thousands of interacting particles in real time. Each pixel in the game world represents a cell that can contain materials like sand, water, oil, or lava. Simple update rules (falling, sliding, mixing, phase changes) are applied in parallel each frame, producing complex effects such as fluid mixing, erosion of terrain, and chain reactions. This pixel-based CA enables dynamic, unpredictable gameplay without heavy physics engines.

### In this notebook you will:
1. Understand how **cellular automata** can model particle physics
2. Implement the **falling sand** mechanics step by step
3. Build a working simulation of granular flow under gravity
4. Visualise particle settling, piling, and flow patterns
5. Lay the groundwork for required project extensions

---

## 0 · Setup

We import NumPy for grid manipulation, Matplotlib for static visualisation, and Matplotlib's animation module together with IPython's HTML display for creating inline animations. We define two cell states — `EMPTY` (0) and `SAND` (1) — and a dark/yellow colour map that visually mimics grains on a dark background.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.animation as animation
from IPython.display import HTML

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Cell states
EMPTY = 0
SAND  = 1

# Colour map: black for empty, sandy yellow for grains
CMAP = ListedColormap(['#1a1a2e', '#e6c86e'])

print('Imports loaded successfully.')

---
## 1 · The Falling Sand Model

The falling sand model is one of the simplest **cellular automata** that produces visually compelling, physically-inspired behaviour. Each cell in a 2D grid is either **empty** or contains a **grain of sand**. Grains obey simple gravity-inspired rules.

### Core Particle Behaviour

At each discrete time step, every particle on the grid evaluates its local neighbourhood and updates according to the following sequence. Updates are computed into a separate array (`new_grid`) so that all movements appear simultaneous and no particle is moved twice in a single step.

1. **Gravity:** If the cell directly below $(r+1, c)$ is empty, the particle moves there. This implements the basic downward pull of gravity.
2. **Diagonal Slide:** If downward movement is blocked (cell below occupied), but either lower-left $(r+1, c-1)$ or lower-right $(r+1, c+1)$ is empty, the particle chooses one of these diagonals. This allows particles to form sloped piles rather than rigid vertical columns.
3. **Lateral Shuffle (optional, for fluids):** For fluid-like materials (water, oil), if neither downward nor diagonal moves are possible, check immediate horizontal neighbours $(r, c-1)$ and $(r, c+1)$. If one side is empty, move there to simulate pooling and spreading.
4. **Settling:** If no moves are available, the particle remains in place and is considered "settled" for this step. Settled particles contribute to local stability and can act as support for particles above.
5. **Material Interactions:** After a move decision, apply any material-specific reactions (e.g., water turning to vapour when in contact with lava, or sand eroding soft terrain beneath).

> **Important implementation detail:** We scan the grid from **bottom to top** so that grains lower in the grid move first, preventing a grain from "falling through" another in the same step.

### Step 1: Single grain update

The `update_grain` function encodes the movement rules for a single sand particle at position `(row, col)`. It checks three possible moves in priority order:

1. **Fall straight down** — if `(row+1, col)` is empty, the grain drops there. This is the dominant behaviour under gravity.
2. **Slide down-left** — if the cell below is blocked but `(row+1, col-1)` is empty, the grain slides diagonally. This creates the sloped sides of piles.
3. **Slide down-right** — symmetric alternative if down-left is also blocked.
4. **Stay** — if all three destinations are occupied, the grain is settled and does not move.

The function modifies `new_grid` in place rather than returning a value. This design allows the full grid update to accumulate all movements into a single output array. We test it by verifying that a grain in mid-air falls down, and that a grain blocked from below slides diagonally.

In [ ]:
def update_grain(grid, new_grid, row, col):
    """Move a single grain of sand according to the falling rules.
    
    Modifies new_grid in place. Returns nothing.
    """
    rows, cols = grid.shape
    # Already at the bottom — cannot fall
    if row >= rows - 1:
        return
    
    # Rule 1: fall straight down
    if new_grid[row + 1, col] == EMPTY:
        new_grid[row, col] = EMPTY
        new_grid[row + 1, col] = SAND
    # Rule 2: slide down-left
    elif col > 0 and new_grid[row + 1, col - 1] == EMPTY:
        new_grid[row, col] = EMPTY
        new_grid[row + 1, col - 1] = SAND
    # Rule 3: slide down-right
    elif col < cols - 1 and new_grid[row + 1, col + 1] == EMPTY:
        new_grid[row, col] = EMPTY
        new_grid[row + 1, col + 1] = SAND
    # Rule 4: stay (already in new_grid)

# Test: grain in mid-air should fall
g = np.zeros((5, 5), dtype=int)
g[1, 2] = SAND
ng = g.copy()
update_grain(g, ng, 1, 2)
assert ng[1, 2] == EMPTY and ng[2, 2] == SAND, 'Grain should fall down'

# Test: grain blocked below should slide
g2 = np.zeros((5, 5), dtype=int)
g2[2, 2] = SAND  # blocker
g2[1, 2] = SAND  # grain to move
ng2 = g2.copy()
update_grain(g2, ng2, 1, 2)
assert ng2[1, 2] == EMPTY, 'Grain should have moved'
assert ng2[2, 1] == SAND or ng2[2, 3] == SAND, 'Grain should slide diagonally'

print('Grain update tests passed.')

### Step 2: Full grid update

The `advance_one_step` function applies `update_grain` to every sand particle in the grid to produce the next time step. The critical implementation detail is the **scan order**: we iterate from the **bottom row upward** (and left to right within each row). This ensures that grains closer to the ground settle first, so that a grain higher up does not "fall through" a grain that hasn't moved yet.

We create a copy of the current grid (`new_grid`) and write all updates there. After the loop, `new_grid` represents the complete state at time $t+1$.

The test verifies two properties:
1. A single grain dropped from the top row eventually reaches the bottom after enough steps.
2. The total number of sand particles is conserved — no grains are created or destroyed during movement.

In [ ]:
def advance_one_step(grid):
    """Update the entire grid by one time step. Returns the new grid."""
    new_grid = grid.copy()
    rows, cols = grid.shape
    # Bottom-to-top scan (skip last row — grains there can't fall)
    for row in range(rows - 2, -1, -1):
        for col in range(cols):
            if new_grid[row, col] == SAND:
                update_grain(grid, new_grid, row, col)
    return new_grid

# Test: a column of sand should settle to the bottom
g3 = np.zeros((10, 10), dtype=int)
g3[0, 5] = SAND
for _ in range(15):
    g3 = advance_one_step(g3)
assert g3[9, 5] == SAND, 'Grain should have reached the bottom'
assert np.sum(g3) == 1, 'Total sand count should be preserved'
print('Full grid update test passed. Sand is conserved.')

### Step 3: Grain source (spawner)

A static grid with pre-placed grains would quickly settle and stop moving. To create a continuous, dynamic simulation we need a **source** that adds new grains over time. The `spawn_grain` function places a single new grain in the top row of the grid, near a specified column with a random horizontal offset controlled by `spread`.

This mimics a hopper or funnel pouring sand from above. The `spread` parameter controls how wide the stream is — a small spread produces a narrow jet that builds a tall, narrow pile, while a large spread distributes grains more evenly. If the target cell is already occupied, the spawn is skipped to prevent overlapping grains.

In [ ]:
def spawn_grain(grid, col=None, spread=5):
    """Add a new grain at the top row near the given column.
    If col is None, spawn at the centre.
    """
    cols = grid.shape[1]
    if col is None:
        col = cols // 2
    offset = np.random.randint(-spread, spread + 1)
    c = max(0, min(cols - 1, col + offset))
    if grid[0, c] == EMPTY:
        grid[0, c] = SAND
    return grid

print('Spawner defined.')

---
## 2 · Running the Simulation

We now combine all three components — grid initialisation, grain spawning, and the CA update rule — into a complete simulation loop. At each of the 400 time steps, we spawn one new grain near the centre of the grid (with a small random horizontal spread of ±3 cells) and then advance the entire grid by one step.

We save snapshots every 50 steps to visualise how the pile grows over time. The simulation starts with an empty grid; grains gradually accumulate, forming a conical pile whose shape is governed by the diagonal-slide rule (which determines the **angle of repose** — the steepest angle at which grains can rest without sliding).

In [ ]:
GRID_SIZE = 80
NUM_STEPS = 400

grid = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)
snapshots = []

for step in range(NUM_STEPS):
    grid = spawn_grain(grid, col=GRID_SIZE // 2, spread=3)
    grid = advance_one_step(grid)
    if step % 50 == 0 or step == NUM_STEPS - 1:
        snapshots.append((step, grid.copy()))

print(f'Simulation complete. Total grains: {int(np.sum(grid))}')

The snapshots below show the pile at regular intervals. Notice how the pile grows from nothing into a roughly triangular (conical) shape. The slope angle is determined by the CA rules: since grains can slide diagonally but not horizontally, the maximum stable slope is approximately 45°. In real granular materials, the angle of repose depends on particle shape, friction, and moisture — the CA rule is a simplified abstraction of this physics.

In [ ]:
fig, axes = plt.subplots(1, len(snapshots), figsize=(3 * len(snapshots), 3))
for ax, (step, snap) in zip(axes, snapshots):
    ax.imshow(snap, cmap=CMAP, vmin=0, vmax=1)
    ax.set_title(f't={step}', fontsize=10)
    ax.axis('off')

fig.suptitle('Sand Pile Formation Over Time', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### Animated simulation

Static snapshots only show discrete moments. An animation reveals the *dynamics* — how grains fall, bounce off the pile, and slide into position. Below we create a smaller $60 \times 60$ grid and run 300 steps, rendering each frame as an image in a JavaScript-based animation widget.

Watch for:
- **Grain cascades** — a newly arriving grain can dislodge others, triggering a chain of slides
- **Pile symmetry** — although each grain lands at a random horizontal offset, the pile remains roughly symmetric due to the unbiased left/right slide rule
- **Surface dynamics** — the top surface of the pile is constantly active as new grains arrive, while the interior is frozen

In [ ]:
ANIM_SIZE = 60
ANIM_STEPS = 300

grid_anim = np.zeros((ANIM_SIZE, ANIM_SIZE), dtype=int)
fig_a, ax_a = plt.subplots(figsize=(5, 5))
img = ax_a.imshow(grid_anim, cmap=CMAP, vmin=0, vmax=1)
ax_a.axis('off')

def update_frame(frame):
    global grid_anim
    grid_anim = spawn_grain(grid_anim, col=ANIM_SIZE // 2, spread=2)
    grid_anim = advance_one_step(grid_anim)
    img.set_data(grid_anim)
    ax_a.set_title(f'Falling Sand — Step {frame}')
    return [img]

plt.close()
anim = animation.FuncAnimation(fig_a, update_frame, frames=ANIM_STEPS,
                                interval=30, blit=True)
HTML(anim.to_jshtml())

---
## 3 · Analysis: Pile Height and Density

In [ ]:
---
## 3 · Analysis: Pile Height and Density

Beyond visual inspection, we can extract quantitative metrics from the simulation to characterise the pile's growth. Two natural measures are:

- **Total grain count** — the number of sand particles on the grid at each step. Since we spawn one grain per step (unless the top row is full), this should increase roughly linearly until the pile reaches the top of the grid.
- **Maximum pile height** — the highest row containing at least one grain. This grows more slowly than the grain count because grains spread laterally as the pile widens.

Below we run a longer simulation (600 steps) and track both metrics. The grain count curve shows the overall mass accumulation rate, while the height curve reveals how quickly the pile grows vertically. A plateau in either curve would indicate that the grid is approaching saturation.

---
## 4 · Your Tasks

The code above implements the basic falling sand mechanics. Your capstone project requires you to add **at least two** of the following features:

### Task A: Material Diversity
Introduce different materials with distinct behaviours:
- **Sand**: stacks (current behaviour)
- **Water**: flows sideways to fill containers, seeks lowest point
- **Oil**: floats on water (lighter density)
- **Lava**: falls like sand but ignites/destroys adjacent materials
- **Gas**: rises upward, disperses horizontally

Each material needs its own update rule and colour.

### Task B: Erosion and Deposition
Allow sand to gradually erode from slopes:
- A grain on a steep slope has a probability of sliding down
- Implement angle-of-repose mechanics
- Show how terrain smooths over time

### Task C: Wind and Fluid Currents
Add directional forces that bias particle movement:
- A wind direction parameter pushes grains left or right as they fall
- Varying wind strength creates different deposition patterns
- Show how wind affects pile shape

### Task D: Complex Structures
Add rigid bodies or platforms:
- Place static obstacle cells that sand cannot pass through
- Implement funnels, hourglasses, or sieves
- Show how obstacles redirect flow

### Discussion points
- Demonstrate results using a GIF or video
- Analyse emergent behaviours (flow patterns, pile shapes)
- Compare simulation to real granular materials
- Discuss how parameter changes affect behaviour

In [ ]:
# ============================================================
# PLACEHOLDER: Implement your extensions below
# ============================================================

# Task A: Material Diversity
# TODO: Add water, oil, lava, gas states and rules

# Task B: Erosion and Deposition
# TODO: Slope-dependent sliding

# Task C: Wind and Currents
# TODO: Directional bias in movement

# Task D: Complex Structures
# TODO: Static obstacles

---
## Recommended Reading & Journal Club

### Foundational References

**1. Bak, P., Tang, C. & Wiesenfeld, K. (1987)**
*Self-organized criticality: An explanation of the 1/f noise.*
Physical Review Letters, 59(4), 381–384. [DOI](https://doi.org/10.1103/PhysRevLett.59.381)
→ The foundational paper on self-organised criticality, using a sandpile model as the key example.

**2. Wolfram, S. (2002)**
*A New Kind of Science.*
Wolfram Media.
→ Comprehensive treatment of cellular automata, including particle-like systems.

---

### Journal Club Papers

**3. Zhu, H. P. et al. (2008)**
*Discrete particle simulation of particulate systems: A review of major applications and findings.*
Chemical Engineering Science, 63(23), 5728–5770. [DOI](https://doi.org/10.1016/j.ces.2008.08.006)
→ Comprehensive review of DEM (Discrete Element Method) for granular flow — the industrial-grade approach to what your CA approximates.

**4. Karolyi, A. & Kertesz, J. (1998)**
*Lattice-gas model of avalanches in a granular pile.*
Physical Review E, 57(1), 852–856. [DOI](https://doi.org/10.1103/PhysRevE.57.852)
→ CA-based avalanche model that bridges the gap between simple sandpile rules and realistic granular dynamics.

**5. Purwins, H. (2020)**
*Noita: a game based on cellular automata.*
Game Developers Conference talk.
→ The game Noita uses falling-sand mechanics with dozens of interacting materials — the gold standard for what your simulation could aspire to.

**6. Jaeger, H. M., Nagel, S. R. & Behringer, R. P. (1996)**
*Granular solids, liquids, and gases.*
Reviews of Modern Physics, 68(4), 1259–1273. [DOI](https://doi.org/10.1103/RevModPhys.68.1259)
→ Classic review of granular physics — explains the real-world phenomena your model approximates.